# MEDUSA: SFT & GRPO Training Pipeline (HF Jobs Edition)

This notebook acts as your orchestrator to fine-tune `Qwen2.5-3B` into an Autonomous Data Engineer.

**Architecture:**
1. Generate expert Chain-of-Thought (CoT) trajectories locally in Colab.
2. Push datasets to the Hugging Face Hub.
3. Dispatch a Hugging Face Compute Job (AWS A10G/L4/A100) to natively run SFT and GRPO, solving latency bottlenecks.
4. Save adapted LoRA weights back to the HF Hub.

In [ ]:
# Install dependencies
!pip install --quiet trl transformers datasets openenv_client peft huggingface_hub

## 1. Authentication
Log into your Hugging Face account to enable Dataset pushing and Jobs submitting.

In [ ]:
from huggingface_hub import login
# Use your WRITE token here
login("hf_YOUR_TOKEN_HERE")

## 2. Fast SFT Synthetic Dataset Generation
We dynamically generate hundreds of random data engineering pipelines including `<think>` reasoning traces.

In [ ]:
!python3 scripts/generate_sft_dataset.py --episodes 100 --out medusa_sft_100_episodes.jsonl

## 3. Dataset Pushing & Smoke Testing
Pushing to Hub allows the decoupled HF Compute Job to instantly load the data locally within the AWS datacenters.

In [ ]:
from datasets import load_dataset

ds = load_dataset("json", data_files="medusa_sft_100_episodes.jsonl")
print(f"Loaded {len(ds['train'])} SFT interaction steps.")

assert len(ds['train']) > 1000, "Error: Too few steps produced. Check generation script!"

# PUSH TO HUB
# ds.push_to_hub("your_username/medusa-sft-trajectories")
print("Uncomment the push_to_hub line above and insert your username!")

## 4. Define the HF Job Script (`train.py`)
We write the actual SFT and GRPO training logic into a file so `huggingface_hub` can submit it to the remote cluster.

In [ ]:
%%writefile train.py
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig, GRPOTrainer, GRPOConfig
from openenv_client import EnvironmentClient
import json
import re

print("Starting Medusa Training Job...")
model_id = "Qwen/Qwen2.5-1.5B-Instruct"
dataset_id = "your_username/medusa-sft-trajectories" # CHANGE ME

# 1. Load Data
ds = load_dataset(dataset_id)
tokenizer = AutoTokenizer.from_pretrained(model_id)

peft_config = LoraConfig(r=16, lora_alpha=32, target_modules=["q_proj", "v_proj"])

# 2. SFT Train
sft_config = SFTConfig(
    output_dir="./medusa-sft",
    max_seq_length=2048,
    per_device_train_batch_size=4,
    learning_rate=2e-5,
    num_train_epochs=1,
    push_to_hub=True, # CRUCIAL for HF Jobs
    hub_model_id="your_username/medusa-qwen-sft" # CHANGE ME
)
# model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.bfloat16)
# trainer = SFTTrainer(model=model, train_dataset=ds['train'], peft_config=peft_config, args=sft_config)
# trainer.train()

# 3. GRPO RL Train (Fast Intra-Networking)
def medusa_reward_function(completions, prompts):
    rewards = []
    # Extremely fast HTTP polling since compute node and Space exist in same data center
    env = EnvironmentClient("anubhavkamal/medusa-env") 
    for completion in completions:
        # (Logic here parses JSON, posts to env, gets reward)
        rewards.append(0.5)
    return rewards

grpo_config = GRPOConfig(
    output_dir="./medusa-grpo",
    learning_rate=5e-6,
    per_device_train_batch_size=2,
    push_to_hub=True,
    hub_model_id="your_username/medusa-qwen-grpo"
)
# grpo_trainer = GRPOTrainer(model=model, reward_funcs=medusa_reward_function, args=grpo_config, train_dataset=ds['train'])
# grpo_trainer.train()

## 5. Submit Training Job
Dispatch the `train.py` script to Hugging Face.

In [ ]:
from huggingface_hub import create_compute_job

# Dispatch job
# job = create_compute_job(
#     script="train.py",
#     hardware="a10g-small",
#     framework="pytorch",
#     env={"HF_TOKEN": "hf_YOUR_TOKEN_HERE"}
# )
# print(f"Job launched: {job.url}")